In [1]:
!pip install -q \
torch==2.3.1 \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
langchain \
langchain-community \
langchain-openai \
langchain-huggingface \
langchain-text-splitters \
chromadb \
pypdf \
pandas \
rank_bm25 \
unstructured \
streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 29.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.8 MB/s eta 0:00:00


In [2]:
documents = None
chunked_documents = None
vector_db = None
vector_retriever = None
bm25_retriever = None

In [3]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever

In [4]:
def load_documents(file_paths):

    documents = []

    for path in file_paths:

        if path.endswith(".pdf"):
            loader = PyPDFLoader(path)

        elif path.endswith(".csv"):
            loader = CSVLoader(path)

        elif path.endswith(".md"):
            loader = UnstructuredMarkdownLoader(path)

        else:
            continue

        docs = loader.load()

        for d in docs:
            d.metadata["source_file"] = path

        documents.extend(docs)

    return documents

In [5]:
import uuid
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_documents(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(documents)

    chunked_docs = []

    for doc in chunks:

        metadata = doc.metadata.copy()

        metadata["chunk_id"] = str(uuid.uuid4())

        chunked_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=metadata
            )
        )

    return chunked_docs

In [6]:
def process_uploaded_documents(files):

    global documents
    global chunked_documents
    global vector_db
    global vector_retriever
    global bm25_retriever
    global KB_SUMMARY

    if files is None or len(files) == 0:
        return "No files uploaded."

    try:

        # Get file paths
        file_paths = [f.name for f in files]

        # Load documents
        documents = load_documents(file_paths)

        if documents is None or len(documents) == 0:
            return "No readable documents found."

        # Chunk documents
        chunked_documents = chunk_documents(documents)

        if chunked_documents is None or len(chunked_documents) == 0:
            return "Document chunking failed."

        # Create knowledge base summary for clarification agent
        KB_SUMMARY = get_kb_summary(chunked_documents)

        # Embedding model
        embedding_model = HuggingFaceEmbeddings(
            model_name="BAAI/bge-small-en"
        )

        # Vector database
        vector_db = Chroma.from_documents(
            documents=chunked_documents,
            embedding=embedding_model
        )

        vector_retriever = vector_db.as_retriever(
            search_kwargs={"k": 3}
        )

        # BM25 keyword retriever
        bm25_retriever = BM25Retriever.from_documents(chunked_documents)
        bm25_retriever.k = 3

        # Debug information (useful for demo)
        return f"""
Documents loaded: {len(documents)}
Chunks created: {len(chunked_documents)}
Vector + BM25 retrievers initialized
"""

    except Exception as e:
        return f"Document processing error: {str(e)}"

In [7]:
import os
import pandas as pd

from langchain_core.documents import Document

from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    UnstructuredMarkdownLoader
)

print("Imports successful")

Imports successful


In [8]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, CSVLoader

print("LangChain imports successful!")

LangChain imports successful!


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [10]:
import numpy as np
import sentence_transformers
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

print("NumPy:", np.__version__)
print("Sentence Transformers loaded")
print("LangChain working")

NumPy: 2.0.2
Sentence Transformers loaded
LangChain working


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
import os

os.environ["OPENAI_API_KEY"] = "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"

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

In [13]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="deepseek/deepseek-chat",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
    max_tokens=800
)

Hybrid Retrieval

In [14]:
def hybrid_retrieve(query, k=4):

    if bm25_retriever is None or vector_retriever is None:
        raise ValueError("Upload documents first using the UI.")

    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    scores = {}

    # BM25 results
    for doc in bm25_docs:
        key = doc.page_content
        scores[key] = scores.get(key, {"doc": doc, "score": 0})
        scores[key]["score"] += 2   # keyword match weight

    # Vector results
    for doc in vector_docs:
        key = doc.page_content
        scores[key] = scores.get(key, {"doc": doc, "score": 0})
        scores[key]["score"] += 1   # semantic match weight

    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)

    return [item["doc"] for item in ranked[:k]]

In [15]:
from langchain.tools import tool as lc_tool

In [16]:
@lc_tool
def factual_qa_tool(query: str) -> dict:
    """
    Use this tool to answer factual questions about the uploaded documents.
    """

    docs = hybrid_retrieve(query)

    context = "\n\n".join([d.page_content for d in docs])

    sources = format_sources(docs)

    prompt = f"""
You are a research assistant.

Use the context below to answer the question.

Context:
{context}

Question:
{query}

Return:

Final Answer:
<direct answer>

Explanation:
<short explanation>
"""

    answer = llm.invoke(prompt).content

    confidence = compute_confidence(docs)

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

In [17]:
@lc_tool
def comparative_analysis_tool(query: str) -> dict:
    """Use this tool when the user asks to compare models or analyze metrics."""

    docs = hybrid_retrieve(query)

    sources = format_sources(docs)


    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""
You are a data analyst.

Use the provided context to compare entities and answer the question.

If structured benchmark data is present, extract the metrics and compare them.

Context:
{context}

Question:
{query}

Return the answer in this format:

Final Answer:
<model with best metric>

Explanation:
<compare the relevant metrics>

Data Used:
<list the values used for comparison>
"""

    answer = llm.invoke(prompt).content

    confidence = compute_confidence(docs)

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

In [18]:
@lc_tool
def summary_tool(query: str) -> dict:
    """
    Use this tool when the user asks to summarize a document or topic.
    """

    docs = hybrid_retrieve(query)

    context = "\n\n".join([d.page_content for d in docs])

    sources = []
    for d in docs:
        source = d.metadata.get("source_file", "unknown")
        page = d.metadata.get("page", None)

        if page is not None:
            sources.append(f"{source} (page {page})")
        else:
            sources.append(source)

    sources = list(set(sources))

    prompt = f"""
Summarize the following content.

Content:
{context}

User request:
{query}

Return a concise summary.
"""

    answer = llm.invoke(prompt).content

    confidence = compute_confidence(docs)

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

In [19]:
def compute_confidence(docs):

    if len(docs) == 0:
        return "Low (0%)"

    # number of retrieved chunks
    chunk_count = len(docs)

    # number of unique documents
    source_count = len(set(d.metadata["source_file"] for d in docs))

    # scoring logic
    score = 0

    # evidence strength
    score += min(chunk_count * 10, 40)

    # cross-source validation
    score += source_count * 20

    # normalize
    percent = min(score, 100)

    if percent >= 80:
        label = "High"
    elif percent >= 60:
        label = "Medium"
    else:
        label = "Low"

    return f"{label} ({percent}%)"

In [20]:
import os
def format_sources(docs):

    source_counts = {}

    for d in docs:

        src = os.path.basename(d.metadata.get("source_file", "unknown"))

        if d.metadata.get("row") is not None:
            src += f" (row {d.metadata['row']})"

        if d.metadata.get("page") is not None:
            src += f" (page {d.metadata['page']})"

        source_counts[src] = source_counts.get(src, 0) + 1


    formatted_sources = []

    total_docs = len(docs)

    for src, count in source_counts.items():

        percent = int((count / total_docs) * 100)

        if percent >= 70:
            level = "High"
        elif percent >= 40:
            level = "Medium"
        else:
            level = "Low"

        formatted_sources.append(
            f"{src} — {level} ({percent}%)"
        )

    return formatted_sources

In [21]:
tools = [
    factual_qa_tool,
    comparative_analysis_tool,
    summary_tool
]

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [23]:
agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI research assistant.

You must choose one tool to answer the user's query.

Available tools:
- factual_qa_tool
- comparative_analysis_tool
- summary_tool

Return ONLY the tool name.
"""
        ),
        ("human", "{query}")
    ]
)

In [24]:
tool_selector = agent_prompt | llm | StrOutputParser()

In [25]:
def run_agent(query):

    tool_name = tool_selector.invoke({"query": query}).strip()

    if "comparative" in tool_name:
        result = comparative_analysis_tool.invoke(query)
        tool_used = "comparative_analysis_tool"

    elif "summary" in tool_name:
        result = summary_tool.invoke(query)
        tool_used = "summary_tool"

    else:
        result = factual_qa_tool.invoke(query)
        tool_used = "factual_qa_tool"

    return {
        "tool_used": tool_used,
        "answer": result
    }

In [26]:
def get_kb_summary(docs, max_chunks=5):

    sample = docs[:max_chunks]

    text = "\n".join([d.page_content[:200] for d in sample])

    prompt = f"""
Summarize the topic/domain of the following documents in 1 sentence.

{text}
"""

    return llm.invoke(prompt).content

In [27]:
def clarification_agent(query):

    prompt = f"""
The assistant answers questions using information from the uploaded documents.

Knowledge base summary:
{KB_SUMMARY}

Determine whether the user's query is ambiguous relative to the knowledge base.

A query is ambiguous ONLY if:
• the metric or criteria is missing
• the entity being compared is unclear
• the question cannot be answered using the uploaded documents

IMPORTANT RULES:

If the user already specifies the metric (examples: latency, accuracy, memory, speed, performance),
then the query is NOT ambiguous.

Example:
"Which model has the lowest latency?" → CLEAR
"Best model in terms of latency" → CLEAR
"Which model is best?" → Ask clarification.

Rules:
- If the query is ambiguous, ask ONE clarification question.
- If the query is clear, respond ONLY with this word:

CLEAR

Do NOT explain your reasoning.
Do NOT add extra sentences.

User query:
{query}
"""

    response = llm.invoke(prompt).content.strip()

    # normalize LLM output
    if response.upper().startswith("CLEAR"):
        return "CLEAR"

    return response

In [28]:
def router_agent(query):

    prompt = f"""
You are an AI system that selects the best tool.

Available tools:
- factual_qa_tool
- comparative_analysis_tool
- summary_tool

Return ONLY the tool name.

Query:
{query}
"""

    return llm.invoke(prompt).content.strip()

In [29]:
def rag_query(query):

    # Step 1 — Clarification agent
    clarification = clarification_agent(query)

    if clarification != "CLEAR":
        return {
            "answer": clarification,
            "sources": [],
            "confidence": "Low",
            "tool": "clarification_agent"
        }

    # Step 2 — Router
    tool_name = router_agent(query)

    # Step 3 — Call tool
    if tool_name == "summary_tool":
        result = summary_tool.invoke({"query": query})

    elif tool_name == "comparative_analysis_tool":
        result = comparative_analysis_tool.invoke({"query": query})

    else:
        result = factual_qa_tool.invoke({"query": query})

    # Step 4 — Attach tool metadata
    result["tool"] = tool_name

    return result

In [ ]:
import gradio as gr


def ask_question(query):

    if vector_retriever is None:
        return "Upload documents first.", "", "", ""

    result = rag_query(query)

    answer = result["answer"]
    sources = "\n".join(result["sources"])
    confidence = result["confidence"]
    tool = result.get("tool", "unknown")

    return answer, sources, confidence, tool


with gr.Blocks() as interface:

    gr.Markdown("# Intelligent Multi-Source Research Assistant")

    files = gr.File(file_count="multiple", label="Upload PDF / CSV / Markdown")

    load_btn = gr.Button("Load Documents")

    status = gr.Textbox(label="Status")

    load_btn.click(
        process_uploaded_documents,
        inputs=files,
        outputs=status
    )

    query = gr.Textbox(label="Ask a question")

    submit = gr.Button("Submit")

    answer = gr.Textbox(label="Answer", lines=6)
    sources = gr.Textbox(label="Sources", lines=4)
    confidence = gr.Textbox(label="Confidence")
    tool = gr.Textbox(label="Tool Used")

    submit.click(
        ask_question,
        inputs=query,
        outputs=[answer, sources, confidence, tool]
    )

interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a7bb22aafbca26a506.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
!pkill -f gradio